Every call is atomic: it either writes all its records or nothing. This page shows the two ways to run a call (in its own transaction, or in yours), what happens when it fails, and which inputs are rejected before anything is written. The examples use SQLite; PostgreSQL behaves the same.

## Setup

The same `users` table as in the Quick Start. Its `email` column is unique, which the failure examples use.

In [1]:
import sqlalchemy as sa

from sqlalchemy_upsert_kit.api import sqlite

metadata = sa.MetaData()

t_user = sa.Table(
    "users",
    metadata,
    sa.Column("id", sa.Integer, primary_key=True),
    sa.Column("name", sa.String(50)),
    sa.Column("email", sa.String(50), unique=True),
)

engine = sa.create_engine("sqlite://")  # an in-memory SQLite database

In [2]:
existing_rows = [
    {"id": 1, "name": "alice", "email": "alice@old.com"},
    {"id": 2, "name": "bob", "email": "bob@old.com"},
    {"id": 3, "name": "cathy", "email": "cathy@old.com"},
]


def reset():
    """Recreate the table with the three existing rows."""
    metadata.drop_all(engine)
    metadata.create_all(engine)
    with engine.begin() as conn:
        conn.execute(t_user.insert(), existing_rows)

In [3]:
records = [
    {"id": 2, "name": "robert", "email": "bob@new.com"},
    {"id": 3, "name": "catherine", "email": "cathy@new.com"},
    {"id": 4, "name": "dave", "email": "dave@new.com"},
]

## Auto-managed mode

By default the call opens a connection, runs in its own transaction, and commits. Nothing else is needed.

In [4]:
reset()
counts = sqlite.insert_or_merge(engine, t_user, records, columns=["email"])
counts

(2, 1)

## User-managed mode

To make the upsert part of a larger transaction, pass your connection as `conn` and your transaction as `trans`. The call then never commits: you decide whether to commit or roll back.

In [5]:
reset()
with engine.connect() as conn:
    with conn.begin() as trans:
        conn.execute(t_user.insert(), [{"id": 10, "name": "admin", "email": "admin@site.com"}])
        counts = sqlite.insert_or_merge(engine, t_user, records, columns=["email"], conn=conn, trans=trans)
        print(f"{counts = }")
    # leaving `with conn.begin()` commits both writes

counts = (2, 1)


In [6]:
with engine.connect() as conn:
    rows = conn.execute(sa.select(t_user).order_by(t_user.c.id)).all()
rows

[(1, 'alice', 'alice@old.com'),
 (2, 'bob', 'bob@new.com'),
 (3, 'cathy', 'cathy@new.com'),
 (4, 'dave', 'dave@new.com'),
 (10, 'admin', 'admin@site.com')]

Several calls can share one transaction. The second call sees the rows of the first: all three records now conflict.

In [7]:
reset()
with engine.connect() as conn:
    with conn.begin() as trans:
        first = sqlite.insert_or_ignore(engine, t_user, records, conn=conn, trans=trans)
        second = sqlite.insert_or_ignore(engine, t_user, records, conn=conn, trans=trans)
print(f"{first = }, {second = }")

first = (2, 1), second = (3, 0)


A rollback of your transaction undoes the upsert too:

In [8]:
reset()
with engine.connect() as conn:
    trans = conn.begin()
    counts = sqlite.insert_or_replace(engine, t_user, records, conn=conn, trans=trans)
    print(f"{counts = }")
    trans.rollback()

counts = (2, 1)


In [9]:
with engine.connect() as conn:
    rows = conn.execute(sa.select(t_user).order_by(t_user.c.id)).all()
rows

[(1, 'alice', 'alice@old.com'),
 (2, 'bob', 'bob@old.com'),
 (3, 'cathy', 'cathy@old.com')]

## When a call fails

A record can be valid and still fail in the database, for example by breaking a constraint. Here Dave's email is already Alice's, which breaks the unique constraint on `email`.

In [10]:
bad_records = [
    {"id": 2, "name": "robert", "email": "bob@new.com"},
    {"id": 4, "name": "dave", "email": "alice@old.com"},
]

In auto-managed mode the error propagates and nothing is written: user 2 is not updated either.

In [11]:
reset()
try:
    sqlite.insert_or_replace(engine, t_user, bad_records)
except sa.exc.IntegrityError as e:
    print(f"{type(e).__name__}: {e.orig}")

IntegrityError: UNIQUE constraint failed: users.email


In [12]:
with engine.connect() as conn:
    rows = conn.execute(sa.select(t_user).order_by(t_user.c.id)).all()
rows

[(1, 'alice', 'alice@old.com'),
 (2, 'bob', 'bob@old.com'),
 (3, 'cathy', 'cathy@old.com')]

In user-managed mode the failure undoes only the call. What you wrote before it stays, the transaction stays usable, and you can retry and commit.

In [13]:
reset()
with engine.connect() as conn:
    with conn.begin() as trans:
        conn.execute(t_user.insert(), [{"id": 10, "name": "admin", "email": "admin@site.com"}])
        try:
            sqlite.insert_or_replace(engine, t_user, bad_records, conn=conn, trans=trans)
        except sa.exc.IntegrityError as e:
            print(f"{type(e).__name__}: {e.orig}")
        counts = sqlite.insert_or_replace(engine, t_user, records, conn=conn, trans=trans)
        print(f"retry: {counts = }")

IntegrityError: UNIQUE constraint failed: users.email
retry: counts = (2, 1)


In [14]:
with engine.connect() as conn:
    rows = conn.execute(sa.select(t_user).order_by(t_user.c.id)).all()
rows

[(1, 'alice', 'alice@old.com'),
 (2, 'robert', 'bob@new.com'),
 (3, 'catherine', 'cathy@new.com'),
 (4, 'dave', 'dave@new.com'),
 (10, 'admin', 'admin@site.com')]

The admin row from before the failure and the retried upsert are both committed.

## Invalid input

Some input can never be upserted correctly. The library checks for it first, and raises `ValueError` before any SQL runs, so nothing is written. One example per rule:

A key that is not a column of the table:

In [15]:
try:
    sqlite.insert_or_ignore(engine, t_user, [{"id": 5, "name": "Eve", "phone": "555"}])
except ValueError as e:
    print(f"{type(e).__name__}: {e}")

ValueError: Records have keys that are not writable columns of 'users': ['phone']


A record without the primary key, or with `None` as the primary key:

In [16]:
try:
    sqlite.insert_or_ignore(engine, t_user, [{"name": "Eve", "email": "eve@new.com"}])
except ValueError as e:
    print(f"{type(e).__name__}: {e}")

ValueError: Records must include the primary key 'id'


In [17]:
try:
    sqlite.insert_or_ignore(engine, t_user, [{"id": None, "name": "Eve", "email": "eve@new.com"}])
except ValueError as e:
    print(f"{type(e).__name__}: {e}")

ValueError: Record 0 has no value for the primary key 'id'


Two records with the same primary key:

In [18]:
try:
    sqlite.insert_or_ignore(
        engine,
        t_user,
        [
            {"id": 5, "name": "Eve", "email": "eve@new.com"},
            {"id": 5, "name": "Eva", "email": "eva@new.com"},
        ],
    )
except ValueError as e:
    print(f"{type(e).__name__}: {e}")

ValueError: Record 1 repeats the primary key id=5


Records with different keys. Every record must have the same keys, so that a missing key is never mistaken for a value:

In [19]:
try:
    sqlite.insert_or_ignore(
        engine,
        t_user,
        [
            {"id": 5, "name": "Eve", "email": "eve@new.com"},
            {"id": 6, "name": "Finn"},
        ],
    )
except ValueError as e:
    print(f"{type(e).__name__}: {e}")

ValueError: Record 1 has keys ['id', 'name'], but record 0 has ['email', 'id', 'name']; all records need the same keys


For `insert_or_merge`, a column in `columns` that the records do not have, so there is no new value to merge:

In [20]:
try:
    sqlite.insert_or_merge(engine, t_user, [{"id": 2, "name": "robert"}], columns=["email"])
except ValueError as e:
    print(f"{type(e).__name__}: {e}")

ValueError: columns ['email'] are not keys of the records, so there is no new value to merge


A connection without its transaction (pass both, or neither):

In [21]:
with engine.connect() as conn:
    try:
        sqlite.insert_or_ignore(engine, t_user, records, conn=conn)
    except ValueError as e:
        print(f"{type(e).__name__}: {e}")

ValueError: Give both conn and trans (user-managed mode) or neither (auto-managed mode)


Empty input is not an error: it writes nothing and returns `(0, 0)`.

In [22]:
sqlite.insert_or_ignore(engine, t_user, [])

(0, 0)

## Other parameters

- `metadata`: the `sa.MetaData` that holds the staging table's definition during the call. A new one by default; passing the target table's own metadata is fine, the staging table is removed from it afterwards.
- `temp_table_name`: the staging table's name. Generated by default, unique per call, so concurrent calls never collide. Leave it unset unless you need a known name.